# Aisle Pacing Ratio: LGBM Ranker Without vs With `user_aisle_pacing_ratio`
Compares the LGBM lambdarank ranker trained on the feature list it used before `user_aisle_pacing_ratio` was added
against the list it uses now. Both models use the screening baseline's settings (`feature_eda_screening.ipynb`), the same 5% user sample,
the same train/validation split and early stopping. There is no hyperparameter tuning.

1. Load a 5% user sample
2. Get both feature lists from `select_features`
3. Split train/validation by user
4. Train both rankers
5. Compare ndcg@5 per order
6. Feature importance of the model with the new feature

In [ ]:
import sys
import os

import bigframes.pandas as bpd
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ndcg_score
import lightgbm as lgb

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.config import source_loc

sys.path.insert(0, source_loc)
from utilities.utility_functions import select_features

### 1. Load a 5% user sample
Sample by user (not by row) so a user's history never lands in both the train and validation split.

In [ ]:
bpd.options.bigquery.project = "instacart-ml-model"
bpd.options.bigquery.location = "US"

bf_full = bpd.read_gbq("instacart-ml-model.ml_model_feature_table.final_ml_features_table")
unique_users = bf_full[['user_id']].drop_duplicates()
sample_users = unique_users.sample(frac=0.05, random_state=42)
df_sample = bf_full.merge(sample_users, on='user_id', how='inner').to_pandas()

print(f"Sample data shape: {df_sample.shape}")
assert 'user_aisle_pacing_ratio' in df_sample.columns
assert df_sample['user_id'].nunique() == len(sample_users)

### 2. Feature lists from the repo's feature selection
`select_features` returns the list the ranker used before the change by default, and the list it uses now with `include_aisle_pacing=True`.

In [ ]:
_, features_without, sort_list = select_features(df_sample)
_, features_with, _ = select_features(df_sample, include_aisle_pacing=True)

assert features_with == features_without + ['user_aisle_pacing_ratio']
print(f"Without: {len(features_without)} features | With: {len(features_with)} features")

### 3. Train/validation split
Same split as the screening baseline: 80% of users train, the rest validate. Sorted by user and order to build the ranker group sizes.

In [ ]:
user_sample = pd.Series(df_sample['user_id'].unique()).sample(frac=0.8, random_state=42)
df_train = df_sample[df_sample['user_id'].isin(user_sample)]
df_val = df_sample[~df_sample['user_id'].isin(user_sample)]

df_train = df_train.sort_values(by=sort_list).reset_index(drop=True)
df_val = df_val.sort_values(by=sort_list).reset_index(drop=True)

assert set(df_train['user_id']).isdisjoint(df_val['user_id'])

#Size of each order defines the groups for the lgbm lambdaRank model
size_array_train = df_train.groupby(sort_list, sort=False).size().to_numpy()
size_array_val = df_val.groupby(sort_list, sort=False).size().to_numpy()

y_train = df_train['label_reordered']
y_val = df_val['label_reordered']

print(df_train.shape, df_val.shape)

### 4. Train both rankers
Screening baseline settings and early stopping of 30 rounds for both models.

In [ ]:
def train_ranker(feature_list):
    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        importance_type='gain',
        device="gpu",
        n_estimators=1000,
        learning_rate=0.05
    )
    ranker.fit(df_train[feature_list], y_train, group=size_array_train,
        eval_set=[(df_val[feature_list], y_val)],
        eval_group=[size_array_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30),
            lgb.log_evaluation(period=50)
        ]
    )
    return ranker

ranker_without = train_ranker(features_without)
ranker_with = train_ranker(features_with)

### 5. ndcg@5 on the validation set
Calculated order by order as in the screening notebook: orders with 1 or fewer reorders are ignored.

In [ ]:
def calc_ndcg_at_5(val_pred, val_true):
    ndcg_score_df = pd.DataFrame({
        'user_id': df_val['user_id'],
        'anchor_order_number': df_val['anchor_order_number'],
        'label': val_true,
        'ranking_score': val_pred
        })

    def process_order(order):
        reorders = order['label'].sum()
        if reorders <= 1:
            return np.nan #ignore orders with no reorders or 1 reorder

        label = order['label'].values.reshape(1, -1)
        ranking_score = order['ranking_score'].values.reshape(1, -1)
        return ndcg_score(label, ranking_score, k=5)

    results = ndcg_score_df.groupby(['user_id', 'anchor_order_number']).apply(process_order)
    results_df = pd.DataFrame(results.to_list(), columns = ['ndcg@5'])

    return results_df

ndcg_without = calc_ndcg_at_5(ranker_without.predict(df_val[features_without]), y_val)['ndcg@5'].mean()
ndcg_with = calc_ndcg_at_5(ranker_with.predict(df_val[features_with]), y_val)['ndcg@5'].mean()

pd.DataFrame({
    'model': ['without user_aisle_pacing_ratio', 'with user_aisle_pacing_ratio'],
    'ndcg@5': [ndcg_without, ndcg_with]
})

In [ ]:
print(f"ndcg@5 difference (with - without): {ndcg_with - ndcg_without:+.5f}")

### 6. Feature importance of the model with `user_aisle_pacing_ratio`
Gain importance with the cumulative percentage, a bar plot, and the rank and share of the new feature.

In [ ]:
feature_importance = pd.DataFrame({
    'feature': features_with,
    'importance': ranker_with.feature_importances_
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

feature_importance['cumulative_sum'] = feature_importance['importance'].cumsum()
feature_importance['cumulative_percentage'] = 100 * feature_importance['cumulative_sum'] / feature_importance['importance'].sum()
display(feature_importance)

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))
sns.barplot(
    data=feature_importance,
    x='importance',
    y='feature',
    color="#0056b3",
    edgecolor="black",
    linewidth=0.5
)
plt.title("Feature Importance (Gain) - Ranker With user_aisle_pacing_ratio", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Importance", fontsize=12, fontweight="medium", labelpad=10)
plt.ylabel("Feature Name", fontsize=12, fontweight="medium", labelpad=10)
sns.despine(top=True, right=True)
plt.show()

In [ ]:
new_feature_row = feature_importance[feature_importance['feature'] == 'user_aisle_pacing_ratio'].iloc[0]
new_feature_rank = feature_importance.index[feature_importance['feature'] == 'user_aisle_pacing_ratio'][0] + 1
new_feature_share = 100 * new_feature_row['importance'] / feature_importance['importance'].sum()

print(f"user_aisle_pacing_ratio ranks {new_feature_rank} of {len(feature_importance)} "
      f"with {new_feature_share:.2f}% of total gain importance")